In [1]:
import scrapy
import pandas as pd
from scrapy.crawler import CrawlerProcess
from scrapy import Request
from scrapy.spiders import Spider
from datetime import datetime

In [2]:
from scrapy.utils.project import get_project_settings

settings = get_project_settings()

settings.set("COOKIES_ENABLED", True, priority="cmdline")
settings.set("ROBOTSTXT_OBEY", False, priority="cmdline")


In [3]:
import os
from mongodb_client import MongoDBClient
mongo_uri = os.getenv("MONGO_URI")
db_name = os.getenv("DB_NAME")
collection = os.getenv("COLLECTION_NAME")

In [4]:
client = MongoDBClient(mongo_uri, db_name, collection)

In [5]:
class LaRazonSpider(Spider):
    name = "larazon"
    allowed_domains = ["www.la-razon.com"]
    start_urls = [
        f"https://www.la-razon.com/tags/feminicidio/page/{i}/" for i in range(10, 12)
    ]
    
    user_agents = [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:120.0) Gecko/20100101 Firefox/120.0"
    ]
    
    custom_headers = {
                    "User-Agent": user_agents[0],
                    "keep_alive": True,
                }
    proxy = "https://3.136.29.104:80"
    
    def __init__(self):
        self.items = []
        self.mongo_client = client
        
    def date_formatter(self, url, date_format="%Y-%m-%d"):
        try:
            url_split = url.split("/")
            date_str = url_split[4] + url_split[5] + url_split[6]
            date_publish = datetime.strptime(date_str, date_format)
            return date_publish
        except Exception as e:
            self.logger.error(f"Error al formatear fecha: {e}")
            return None
    
    def tittle_formatter(self, title):
        try:
            title = title.replace("“", '"')
            title = title.replace("”", '"')
            return title
        except Exception as e:
            self.logger.error(f"Error al formatear título: {e}")
            return title
    
    def tag_formatter(self, tags):
        try:
            list_tags = [t.full_text.lower() for t in tags]
            return list_tags
        except Exception as e:
            self.logger.error(f"Error al formatear tags: {e}")
            return tags
    
    def section_formatter(self, url):
        try:
            url_split = url.split("/")
            section = url_split[3]
            return section
        except Exception as e:
            self.logger.error(f"Error al formatear sección: {e}")
            return url

    def body_formatter(self, body):
        try:
            new_body = [
                b.strip()
                .replace("\xa0", " ")
                .replace("\ufeff", " ")
                .replace("“", '"')
                .replace("”", '"')
                .replace("\u200b", " ")
                for b in new_body
            ]
            new_body = [b for b in new_body if b != " "]
            return new_body
        except Exception as e:
            self.logger.error(f"Error al formatear cuerpo: {e}")
            return body

    def start_requests(self):
        for url in self.start_urls:
            self.logger.info(f"Enviando request a: {url}")
            yield Request(url=url, callback=self.parse_response, headers=self.custom_headers, meta={"proxy": self.proxy})
            
    
    def parse_response(self, response):
        self.logger.info(f"Recibida respuesta: {response.url}")
        try:
            self.logger.info(f"Cuerpo de la respuesta: {response.xpath('//div[@id="lr-main"]')}")
            noticias = response.xpath('(//div[@class="articles-list"])[1]//div[@class="article-meta "]/a').getall()
            
            self.logger.info(f"Total noticias encontradas: {len(noticias)}")
            for noticia in noticias:
                self.logger.info(f"Enviando request a: {noticia}")
                yield Request(url=noticia, callback=self.parse_news, headers=self.custom_headers, meta={"proxy": self.proxy})
        except Exception as e:
            self.logger.error(f"Error al procesar la respuesta JSON: {e}")
            return
    
    def parse_news(self, response):
        title = response.xpath("(//h1[@class='title'])[1]/text()").get()
        item = {}
        item["url"] = response.url
        item["title"] = self.tittle_formatter(title)
        tags = response.xpath('//div[@class="lr-tags-cloud-block"]//a[contains(@href, "tag")]/li/text()').get()
        item["tags"] = self.tag_formatter(tags)
        item["section"] = self.section_formatter(response.url)
        body = [
            p.xpath("string(.)").get()
            for p in response.xpath("//div[contains(@class, 'article-body')]/p")
        ]
        item["body"] = self.body_formatter(body)
        item["date_published"] = self.date_formatter(response.url, "%d/%m/%Y")
        item["source"] = "larazon"
        self.items.append(item)
        self.logger.info(f"Noticia agregada: {item['title']}")
    
    def close(self, reason):
        file_path = "datos.txt"
        self.logger.info("Guardando datos en archivo de texto")

        try:
            with open(file_path, "w", encoding="utf-8") as f:
                for item in self.items:
                    f.write(f"{item}\n")
            self.logger.info(f"Datos guardados en {file_path}")
        except Exception as e:
            self.logger.error(f"Error al guardar en el archivo: {e}")

        self.logger.info(f"Spider cerrado por la razón: {reason}")

In [6]:
process = CrawlerProcess(settings)
process.crawl(LaRazonSpider)
process.start()

INFO:scrapy.utils.log:Scrapy 2.11.2 started (bot: scrapybot)
2025-03-10 17:59:56 [scrapy.utils.log] INFO: Scrapy 2.11.2 started (bot: scrapybot)
INFO:scrapy.utils.log:Versions: lxml 5.3.0.0, libxml2 2.11.7, cssselect 1.2.0, parsel 1.9.1, w3lib 2.2.1, Twisted 24.7.0, Python 3.12.1 (tags/v3.12.1:2305ca5, Dec  7 2023, 22:03:25) [MSC v.1937 64 bit (AMD64)], pyOpenSSL 24.2.1 (OpenSSL 3.3.2 3 Sep 2024), cryptography 43.0.1, Platform Windows-10-10.0.19045-SP0
2025-03-10 17:59:56 [scrapy.utils.log] INFO: Versions: lxml 5.3.0.0, libxml2 2.11.7, cssselect 1.2.0, parsel 1.9.1, w3lib 2.2.1, Twisted 24.7.0, Python 3.12.1 (tags/v3.12.1:2305ca5, Dec  7 2023, 22:03:25) [MSC v.1937 64 bit (AMD64)], pyOpenSSL 24.2.1 (OpenSSL 3.3.2 3 Sep 2024), cryptography 43.0.1, Platform Windows-10-10.0.19045-SP0
INFO:scrapy.addons:Enabled addons:
[]
2025-03-10 17:59:56 [scrapy.addons] INFO: Enabled addons:
[]

It is also the default value. In other words, it is normal to get this warning if you have not defined a val